In [1]:
# Enable auto-reloading of utils.py so you don't have to restart the kernel
%load_ext autoreload
%autoreload 2

import json
import numpy as np
from sentence_transformers import SentenceTransformer

# Binary-relevance metrics only — no cache, no live SPARQL needed
from utils import (
    avg_prf_at_k,
    avg_mrr_at_k,
    avg_hits_at_k,
    avg_jaccard_at_k,   # ← added
    avg_ndcg_at_k,
)

QWEN_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE      = 32
TOPKS           = [3, 5, 10, 20]

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the triplet dataset — positives were already selected by Jaccard threshold
file_path = "hard_triplets_tfidf_wikidata.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Successfully loaded {len(data)} query groups from the JSON file.")

Successfully loaded 186 query groups from the JSON file.


In [3]:
unique_queries = set()
eval_items     = []

for item in data:
    anchor    = item["anchor_query"]
    positives = item["positive_queries"]
    negatives = item["negative_queries"]

    # Pool = all candidates the model must rank
    pool = positives + negatives

    unique_queries.add(anchor)
    unique_queries.update(positives)
    unique_queries.update(negatives)

    eval_items.append((anchor, pool, positives))

queries_list = list(unique_queries)
print(f"Extracted {len(eval_items)} evaluation items.")
print(f"Found {len(queries_list)} unique queries for the embedding vocabulary.")

Extracted 186 evaluation items.
Found 2227 unique queries for the embedding vocabulary.


In [4]:
# ==========================================================================
# Build Binary Relevance Maps
# ==========================================================================
# Rationale: positive_queries were selected at build time using
#   Jaccard(answers(anchor), answers(candidate)) >= threshold
# So membership in positive_queries IS the relevance label.
# No SPARQL endpoint, no answer cache, and no online Jaccard computation needed.

binary_relevance_per_item = []

for _, pool, positives in eval_items:
    positive_set  = set(positives)
    relevance_map = {q: (1.0 if q in positive_set else 0.0) for q in pool}
    binary_relevance_per_item.append(relevance_map)

print(f"Built binary relevance maps for {len(binary_relevance_per_item)} items.")

Built binary relevance maps for 186 items.


In [5]:
# ==========================================================================
# Dense Model Helper Functions
# ==========================================================================

def build_embedding_map(model, queries, batch_size=32, normalize_embeddings=True):
    """Encodes all queries exactly once and stores them in a dict."""
    embedding_map = {}
    for start in range(0, len(queries), batch_size):
        batch      = queries[start : start + batch_size]
        embeddings = model.encode(
            batch,
            convert_to_numpy=True,
            normalize_embeddings=normalize_embeddings,
        )
        for query, emb in zip(batch, embeddings):
            embedding_map[query] = emb
    return embedding_map


def evaluate_dense_model(model_name, model, queries, items, batch_size=32):
    """Encodes all queries once, then ranks each pool by dot-product similarity."""
    print(f"\nEvaluating dense model: {model_name}")

    embedding_map = build_embedding_map(model, queries, batch_size)
    sorted_pools  = []

    for anchor_query, pool, _ in items:
        pool_embs  = np.vstack([embedding_map[q] for q in pool])
        anchor_emb = embedding_map[anchor_query]
        scores     = pool_embs @ anchor_emb
        order      = np.argsort(-scores)
        sorted_pools.append([pool[i] for i in order])

    return sorted_pools

In [7]:
# ==========================================================================
# Run Qwen and Print Metrics
# ==========================================================================

# 1. Load model
print(f"Loading {QWEN_MODEL_NAME}...")
qwen_model = SentenceTransformer(QWEN_MODEL_NAME)

# 2. Rank pools
qwen_sorted_pools = evaluate_dense_model(
    model_name = QWEN_MODEL_NAME,
    model      = qwen_model,
    queries    = list(unique_queries),
    items      = eval_items,
    batch_size = BATCH_SIZE,
)

# 3. Print results
print("\n" + "=" * 60)
print(f"FINAL EVALUATION RESULTS: {QWEN_MODEL_NAME}")
print("=" * 60)

# Binary relevance — membership in positive_queries is the sole label
print("\n[Binary relevance: Annotated Positives]")
print(f"{'':5} {'P':>7} {'R':>7} {'F1':>7} {'MRR':>7} {'Hits':>7} {'avgJacc':>9} {'nDCG':>7}")
print("-" * 62)
for k in TOPKS:
    p, r, f1 = avg_prf_at_k(qwen_sorted_pools, eval_items, k)
    mrr      = avg_mrr_at_k(qwen_sorted_pools, eval_items, k)
    hits     = avg_hits_at_k(qwen_sorted_pools, eval_items, k)
    ndcg     = avg_ndcg_at_k(qwen_sorted_pools, k, binary_relevance_per_item)
    avg_jacc = avg_jaccard_at_k(qwen_sorted_pools, k, binary_relevance_per_item)
    
    print(f"@{k:<4} {p:>7.4f} {r:>7.4f} {f1:>7.4f} {mrr:>7.4f} {hits:>7.4f} {avg_jacc:>9.4f} {ndcg:>7.4f}")

Loading Qwen/Qwen3-Embedding-0.6B...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 984.67it/s, Materializing param=norm.weight]                               



Evaluating dense model: Qwen/Qwen3-Embedding-0.6B

FINAL EVALUATION RESULTS: Qwen/Qwen3-Embedding-0.6B

[Binary relevance: Annotated Positives]
            P       R      F1     MRR    Hits   avgJacc    nDCG
--------------------------------------------------------------
@3     0.4337  0.1268  0.1808  0.5367  0.6505    0.4337  0.4345
@5     0.4172  0.1958  0.2418  0.5553  0.7312    0.4183  0.4330
@10    0.4011  0.3780  0.3482  0.5675  0.8226    0.4043  0.4619
@20    0.3798  0.7240  0.4430  0.5782  0.9731    0.3851  0.5798
